# Mapping Physicochemical Features onto 2D Interface Planes for Protein–Protein Binding Affinity Prediction

**Corso:** Advanced Machine Learning for Physics (A.A. 2025/2026)
**Istituzione:** Sapienza Università di Roma
**Candidata:** Chiara Ritorto

Questo notebook raccoglie ed esegue, in ordine, le fasi del progetto descritte nel report (`report/report.pdf`).
Il calcolo pesante (estrazione delle interfacce, ChimeraX, Zernike, training della CNN) non viene rilanciato qui:
il notebook carica i risultati già prodotti e ne mostra tabelle e grafici, in modo che sia eseguibile ovunque
(compreso Google Colab) senza dipendenze da ChimeraX o da un cluster HPC.


## 0. Setup dell'ambiente

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/chiararitorto/ppb-affinity.git"
    REPO_DIR = Path("/content/ppb-affinity")
    if not REPO_DIR.exists():
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
else:
    # Esecuzione locale: assume che il notebook sia nella root del repository
    REPO_DIR = Path.cwd()

sys.path.append(str(REPO_DIR))
print(f"Root del repository: {REPO_DIR}")


Root del repository: /content/ppb-affinity


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

pd.set_option("display.max_columns", 20)
print("Librerie caricate correttamente.")


Librerie caricate correttamente.


## 1. Dataset

Il dataset di partenza è l'*Affinity Benchmark v5.5*, con PDB ID, catene di ligando/recettore e costante di
dissociazione K$_D$ per ciascun complesso.

In [3]:
df_affinity = pd.read_csv(REPO_DIR / "data" / "affinity_dataset.csv", sep=";")
df_affinity.columns = [c.strip() for c in df_affinity.columns]
print(f"Numero di complessi nel dataset: {len(df_affinity)}")
df_affinity.head()


Numero di complessi nel dataset: 207


,Unnamed: 0,PDB,Source Data Set,Model,Mutations,Ligand Chains,Receptor Chains,Ligand Name,Receptor Name,KD(M),Affinity Method,Structure Method,Temperature(K),Resolution(Å),PDB PubMed ID,PDB Release Date,Affinity PubMed ID,Affinity Release Date
0,0,1KTZ,Affinity Benchmark v5.5,NaN,NaN,A,B,TGF-beta,TGF-beta receptor,2.9e-07,NaN,X-RAY DIFFRACTION,"300,15",2.15,11850637.0,2002-02-27,16300789,2006 Jan 6
1,1,1JTG,Affinity Benchmark v5.5,NaN,NaN,B,A,beta-lactamase inhibitor protein,beta-lactamase TEM-1,4e-10,NaN,X-RAY DIFFRACTION,"300,15",1.73,11573088.0,2001-10-17,9890878,1999 Jan 5
2,2,4ETQ,Affinity Benchmark v5.5,NaN,NaN,"H,L",C,LA5,vaccinia D8L IMV,1.8e-10,NaN,X-RAY DIFFRACTION,NaN,2.10,22623786.0,2012-06-06,23152530,2013 Feb
3,3,1DE4,Affinity Benchmark v5.5,NaN,NaN,"A,B","C,F",hemochromatosis protein HFE,Transferrin receptor ectodom.,6.8e-08,NaN,X-RAY DIFFRACTION,NaN,2.80,10638746.0,2000-01-19,11800564,2001 Oct 19
4,4,1JPS,Affinity Benchmark v5.5,NaN,NaN,"H,L",T,Fab D3H44,Tissue factor,1e-10,NaN,X-RAY DIFFRACTION,NaN,1.85,11601848.0,2002-02-03,11307801,2001 Mar


## 2. Task 1 — Interface Identification and Surface Patch Extraction

Pipeline in `src/` (moduli `pdb.py`, `interface.py`, `surface.py`, `chimerax.py`, `dataset.py`, `pipeline.py`),
eseguita tramite `scripts/task1/run_interface_extraction.py`. Identifica i residui di interfaccia
(cutoff 5.0 Å) ed estrae le patch di superficie molecolare (backend ChimeraX, `full_chain_then_filter`).

Il risultato per ciascun complesso è salvato in `outputs/task1/<indice>_<PDB_ID>/`. Qui carichiamo il
`manifest.csv` riassuntivo e un complesso di esempio (1KTZ).

In [4]:
manifest = pd.read_csv(REPO_DIR / "outputs" / "task1" / "manifest.csv")
print(f"Complessi nel manifest: {len(manifest)}")
print(manifest['status'].value_counts())
manifest.head()


Complessi nel manifest: 207
status
ok    207
Name: count, dtype: int64


,row_index,pdb_id,status,output_dir,all_atoms,ligand_atoms,receptor_atoms,ligand_interface_residues,receptor_interface_residues,residue_contacts,ligand_patch_atoms,receptor_patch_atoms,ligand_surface_points,receptor_surface_points,ligand_full_surface_points,receptor_full_surface_points,error
0,0,1KTZ,ok,outputs/task1/00000_1KTZ,1493,653,840,9,13,30,90,101,12528,12875,84247,98215,NaN
1,1,1JTG,ok,outputs/task1/00001_1JTG,6507,1236,2022,33,35,93,294,274,44256,34694,158542,206531,NaN
2,2,4ETQ,ok,outputs/task1/00002_4ETQ,10235,3287,1793,28,30,70,262,247,44034,32294,462643,190159,NaN
3,3,1DE4,ok,outputs/task1/00003_1DE4,24306,3063,10074,25,26,59,217,221,28958,46145,344169,1172938,NaN
4,4,1JPS,ok,outputs/task1/00004_1JPS,4858,3247,1611,27,24,71,236,188,28916,23756,364064,181259,NaN


### 2.1 Selezione dei dimeri

Dal dataset completo vengono selezionati i soli complessi dimerici (una catena per ligando e per recettore),
secondo lo stesso criterio usato in `scripts/task1/filter_dimers.py`.

In [5]:
is_dimer = (df_affinity['Ligand Chains'].astype(str).str.len() == 1) & \
           (df_affinity['Receptor Chains'].astype(str).str.len() == 1)
dimer_ids = set(df_affinity.loc[is_dimer, 'PDB'].astype(str).str.strip())

manifest_dimers = manifest[manifest['pdb_id'].isin(dimer_ids)].drop_duplicates(subset='pdb_id')
print(f"Complessi dimerici trovati: {len(manifest_dimers)}")

   # Nota: 1NVU compare due volte nel dataset originale (catene ligando Q e R,
   # varianti puntiformi dello stesso complesso, stesso recettore S). La pipeline
   # indicizza l'output per PDB ID, quindi ne conserva una sola cartella su disco;
   # da qui la deduplica esplicita sopra.


Complessi dimerici trovati: 121


### 2.2 Esempio: complesso 1KTZ

In [ ]:
example_dir = REPO_DIR / "outputs" / "task1" / "00000_1KTZ"

with open(example_dir / "metadata.json") as f:
    meta_1ktz = json.load(f)

print(json.dumps(meta_1ktz["counts"], indent=2))


In [ ]:
ligand_patch = pd.read_csv(example_dir / "ligand_surface_patch.csv")
receptor_patch = pd.read_csv(example_dir / "receptor_surface_patch.csv")

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(ligand_patch["x"], ligand_patch["y"], ligand_patch["z"], s=2, alpha=0.5, label="ligando (A)")
ax.scatter(receptor_patch["x"], receptor_patch["y"], receptor_patch["z"], s=2, alpha=0.5, label="recettore (B)")
ax.set_title("Patch di interfaccia — complesso 1KTZ")
ax.legend()
plt.tight_layout()
plt.show()


### 2.3 Statistiche aggregate sui dimeri

In [ ]:
cols = ['ligand_interface_residues', 'receptor_interface_residues', 'residue_contacts',
        'ligand_patch_atoms', 'receptor_patch_atoms',
        'ligand_surface_points', 'receptor_surface_points', 'all_atoms']

manifest_dimers[cols].describe().T[['mean', 'std', 'min', '50%', 'max']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))

axes[0].hist(manifest_dimers['ligand_interface_residues'], bins=15, alpha=0.7, label='ligando', color='#4C72B0')
axes[0].hist(manifest_dimers['receptor_interface_residues'], bins=15, alpha=0.7, label='recettore', color='#DD8452')
axes[0].set_xlabel('Numero di residui di interfaccia')
axes[0].set_ylabel('Numero di complessi')
axes[0].set_title("Dimensione dell'interfaccia (dimeri)")
axes[0].legend(fontsize=8)

axes[1].hist(manifest_dimers['residue_contacts'], bins=15, color='#55A868')
axes[1].set_xlabel('Numero di contatti residuo-residuo')
axes[1].set_ylabel('Numero di complessi')
axes[1].set_title('Contatti di interfaccia (dimeri)')

plt.tight_layout()
plt.show()


## 3. Task 2 — Mappatura della complementarità tramite descrittori di Zernike

Script di produzione: `scripts/task2/run_task2_parallel.py`, eseguito su CINECA Leonardo per tutti i 120 dimeri.
Per ciascun complesso i punti di superficie di ligando e recettore (Task 1) vengono sottocampionati con stride 20, espansi
in descrittori locali di Zernike (ordine 20, raggio 6.0 Å, `verso = +1` per il ligando e `-1` per il recettore), e confrontati
con un matching globale nello spazio dei descrittori (`scipy.spatial.distance.cdist`), simmetrico andata + ritorno.

Vedi la Sezione "Task 2" del report per la spiegazione completa dell'algoritmo e del costo computazionale su HPC.

In [ ]:
task2_example_dir = REPO_DIR / "outputs" / "task2" / "00000_1KTZ"

zernike_1ktz = pd.read_csv(task2_example_dir / "zernike_complementarity_symmetric.csv")
print(f"Righe totali: {len(zernike_1ktz)}")
print(zernike_1ktz['source'].value_counts())
zernike_1ktz.head()

### 3.1 Distribuzione dello score di complementarità

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for source, color in [("ligand", "#4C72B0"), ("receptor", "#DD8452")]:
    subset = zernike_1ktz[zernike_1ktz["source"] == source]
    ax.hist(subset["complementarity_score"], bins=25, alpha=0.6, label=source, color=color)

ax.set_xlabel("Complementarity score (= -distanza Zernike)")
ax.set_ylabel("Numero di punti")
ax.set_title("Distribuzione dello score di complementarità — 1KTZ")
ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Visualizzazione spaziale dei punti campionati, colorati per score

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

sc = ax.scatter(
    zernike_1ktz["x"], zernike_1ktz["y"], zernike_1ktz["z"],
    c=zernike_1ktz["complementarity_score"], cmap="viridis", s=15
)
ax.set_xlabel("x (Å)")
ax.set_ylabel("y (Å)")
ax.set_zlabel("z (Å)")
ax.set_title("Punti campionati e score di complementarità — 1KTZ")
fig.colorbar(sc, ax=ax, label="complementarity score", shrink=0.6)
plt.tight_layout()
plt.show()

## 4. Task 3 — Costruzione dei piani di complementarità 2D

*(sezione da popolare non appena saranno caricati gli output della Task 3: mappe .npy, sommario PCA, analisi Lennard-Jones)*

## 5. Task 4 — Predizione dell'affinità tramite CNN

*(sezione da popolare non appena saranno caricati i pesi del modello e le predizioni out-of-fold)*

## 6. Discussione e conclusioni

*(da completare in linea con la Sezione "Discussione e limiti" del report)*